# 04 Train Healthcare LoRA Adapter

This notebook trains the `healthcare` standalone PEFT LoRA adapter and writes it to `adapters/healthcare/`.

```mermaid
flowchart LR
    A[training_data/healthcare.json] --> B[Qwen chat template]
    B --> C[PEFT LoRA training]
    C --> D[adapters/healthcare/]
    C --> E[MLflow run]
```

The adapter is not merged into the base model. vLLM loads it later as a runtime LoRA adapter.

In [ ]:
from pathlib import Path
import os
import sys

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "PROJECT_SPEC.md").exists() else CURRENT_DIR.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
sys.path.append(str(PROJECT_ROOT))

from training.config import DEFAULT_CONFIG

cfg = DEFAULT_CONFIG.copy()
cfg["data_dir"] = str((NOTEBOOK_DIR / cfg["data_dir"]).resolve())
cfg["output_dir"] = str((NOTEBOOK_DIR / cfg["output_dir"]).resolve())
os.environ["DATA_DIR"] = cfg["data_dir"]
os.environ["ADAPTER_DIR"] = cfg["output_dir"]
os.environ.setdefault("MLFLOW_EXPERIMENT_NAME", cfg["experiment_name"])

from llmops_demo.settings import ensure_dirs, settings

settings_cfg = settings()
ensure_dirs(Path(cfg["data_dir"]), Path(cfg["output_dir"]))

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {cfg['data_dir']}")
print(f"Adapter output dir: {cfg['output_dir']}")
print(f"Base model: {settings_cfg.base_model}")
print(f"Adapters: {settings_cfg.adapters}")

## Preflight

Expected inputs:

- `training_data/healthcare.json`
- access to the base model configured by `BASE_MODEL`
- CUDA GPU recommended for practical runtime

In [ ]:
dataset_path = Path(cfg["data_dir"]) / "healthcare.json"
adapter_path = Path(cfg["output_dir"]) / "healthcare"
print(f"Dataset exists: {dataset_path.exists()} - {dataset_path}")
print(f"Adapter output: {adapter_path}")

## Train

This calls the shared training entry point, logs to MLflow, and saves the standalone PEFT adapter.

In [ ]:
from training.train_lora import train_adapter

train_adapter("healthcare", settings_cfg)

## Verify adapter files

Expected output: PEFT adapter files under `adapters/healthcare/`.

In [ ]:
for path in sorted((Path(cfg["output_dir"]) / "healthcare").glob("*")):
    print(path)